## Importing necessary libraries

In [ ]:
print('Hello world')
import numpy as np
import tensorflow as tf
import random
import re
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Hello world


### Loading dataset

In [79]:
def load_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

text = load_text("../data/faq.txt")
sentences = text.split("\n")

### Creating Vocabulary

In [81]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

total_words = len(tokenizer.word_index)
total_words


282

In [82]:
train_seq = tokenizer.texts_to_sequences(sentences)

In [83]:
train_seq[:2]

[[93, 1, 13], [11, 7, 1, 12, 42, 15, 43, 53, 68, 13, 147, 148]]

In [85]:
max_len = max(len(x) for x in train_seq)+1
max_len


58

### Dataset Generation

In [86]:

def data_generator(sequences, max_len):
    input_sequences = []
    # creating ngrams
    for seq in sequences:
        for i in range(1, len(seq)):
            n_gram_sequence = seq[:i+1]
            input_sequences.append(n_gram_sequence)
    # padding
    input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
    X = input_sequences[:,:-1]
    y = input_sequences[:,-1]
    return X,y
    

In [87]:
X_train, y_train = data_generator(train_seq, max_len)

In [88]:
X_train.shape, y_train.shape

((863, 57), (863,))

In [89]:
model = Sequential([
    Embedding(total_words+1, 100, input_length=max_len-1),
    LSTM(64),
    Dense(total_words+1, activation='softmax')
])
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/Users/prince-khatri/Documents/Learning/venv_3_10/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [91]:
model.fit(X_train,y_train, epochs=100)

Epoch 1/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9293 - loss: 0.4545
Epoch 2/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9258 - loss: 0.4454
Epoch 3/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9305 - loss: 0.4366
Epoch 4/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9305 - loss: 0.4291
Epoch 5/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9270 - loss: 0.4215
Epoch 6/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9293 - loss: 0.4128
Epoch 7/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9305 - loss: 0.4056
Epoch 8/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9305 - loss: 0.3989
Epoch 9/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9305 - loss: 0.3927
Epoch 10/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9351 - loss: 0.3850
Epoch 11/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9351 - loss: 0.3795
Epoch 12/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

In [154]:
def predictor(model, tokenizer, text, max_size, top_k):
    # seq generation
    text_seq = tokenizer.texts_to_sequences([text])[0]
    # padding
    text_padded = pad_sequences([text_seq], maxlen=max_size, padding='pre')

    prob = model.predict(text_padded, verbose=0)[0]

    top_k_indices = np.argsort(prob)[-top_k:]
    top_k_prob = prob[top_k_indices]
    top_k_prob /= top_k_prob.sum()

    predicted = np.random.choice(top_k_indices,p=top_k_prob)
    # print(tokenizer.index_word.get(top_k_indices[0]),tokenizer.index_word.get(top_k_indices[1]),tokenizer.index_word.get(top_k_indices[2]))
    return tokenizer.index_word.get(predicted,"")







    

In [165]:
text = "what is course"
for _ in range(4):
    text+= " "+predictor(model, tokenizer,text , max_len, 3)
print(text)
    

what is course fee for earlier months
